# Ch 6　State 設計：schema、reducer 與 annotation

> **本 notebook 完全不需要 API key**——state 與 reducer 是純機制，最適合動手玩到滾瓜爛熟。

In [ ]:
!uv pip install -q langgraph

## 6.2 vs 6.3　覆蓋 reducer vs 累加 reducer，眼見為憑

In [1]:
from typing import Annotated, TypedDict
from operator import add

# foo 沒貼 reducer = 預設「覆蓋」；bar 貼了 operator.add = 「累加」。
class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]

In [2]:
# 我們不建整張圖，直接借 langgraph 的更新邏輯做兩個小節點看效果。
from langgraph.graph import StateGraph, START, END

def node_1(state): return {"foo": 2, "bar": ["hi"]}
def node_2(state): return {"foo": 99, "bar": ["bye"]}   # foo 會被覆蓋成 99，bar 會累加上 bye

b = StateGraph(State)
b.add_node("n1", node_1); b.add_node("n2", node_2)
b.add_edge(START, "n1"); b.add_edge("n1", "n2"); b.add_edge("n2", END)
g = b.compile()

print(g.invoke({"foo": 1, "bar": []}))
# 預期：{'foo': 99, 'bar': ['hi', 'bye']}
# foo 一路被覆蓋（只剩最後的 99）；bar 被一路累加（hi 還在）。這就是 reducer 的差別。

{'foo': 99, 'bar': ['hi', 'bye']}


## 6.4–6.5　對話用 add_messages，以及 MessagesState

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

# add_messages：新訊息就附加；但帶「既有 ID」的訊息會「覆蓋」而非重複附加——這對人工改寫超重要。
existing = [HumanMessage(content="你好", id="m1"), AIMessage(content="哈囉！", id="m2")]

# 情境 A：附加一則全新訊息
print("附加新訊息：")
for m in add_messages(existing, [HumanMessage(content="幫我訂位", id="m3")]):
    print("  ", m.id, m.content)

# 情境 B：送一則「同 ID」的訊息 -> 覆蓋 m2，而不是又多一則
print("以同 ID 覆蓋既有訊息：")
for m in add_messages(existing, [AIMessage(content="（已改寫）哈囉，很高興為您服務！", id="m2")]):
    print("  ", m.id, m.content)

In [ ]:
from langgraph.graph import MessagesState

# 最常見的起手式：繼承 MessagesState 拿到 messages 通道（已配好 add_messages），再加自己的欄位。
class AgentState(MessagesState):
    documents: list[str]      # 額外欄位（這個用預設覆蓋）

print(AgentState.__annotations__)   # 看得到 messages + documents 兩個欄位

## 重點回顧

沒貼 reducer = 覆蓋；貼了 = 用你指定的方式合併。對話一律用 `add_messages`（會聰明處理「改寫既有訊息」）。下一章我們會揭露：你貼的 reducer，在引擎裡的真身其實是「channel 的更新函數」。